In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Qiskit Function od Qedmy
*Viz [referenci API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Qiskit Functions jsou experimentální funkce dostupné pouze uživatelům plánů IBM Quantum&reg; Premium Plan, Flex Plan a On-Prem (prostřednictvím IBM Quantum Platform API). Jsou ve stavu preview verze a mohou se měnit.

## Přehled
Přestože se kvantové procesory (QPU) v posledních letech výrazně zlepšily, chyby způsobené hlukem a nedokonalostmi stávajícího hardwaru zůstávají klíčovou výzvou pro vývojáře kvantových algoritmů. Jak se obor přibližuje kvantovým výpočtům v měřítku užitečnosti, které nelze klasicky ověřit, stávají se řešení pro eliminaci šumu se zaručenou přesností čím dál důležitějšími. K překonání této výzvy vyvinula Qedma metodu Quantum Error Mitigation (QESEM), která je bezešvě integrována na IBM Quantum Platform jako [Qiskit Function](/guides/functions).

S QESEM můžeš spouštět své kvantové obvody na hlučných QPU a získávat vysoce přesné výsledky bez chyb s vysoce efektivní spotřebou QPU času, blízkou fundamentálním limitům. QESEM k tomu využívá sadu proprietárních metod vyvinutých Qedmou pro charakterizaci a redukci chyb. Techniky redukce chyb zahrnují optimalizaci Gate, transpilaci s ohledem na šum, potlačení chyb (ES) a nestrannou mitigaci chyb (EM). Díky kombinaci těchto metod založených na charakterizaci mohou uživatelé dosáhnout spolehlivých výsledků bez chyb pro obecné kvantové obvody s velkým objemem, čímž se odemykají aplikace, které by jinak nebylo možné realizovat.

Úplný popis základních komponent i demonstraci v měřítku užitečnosti najdeš v článku [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Popis
Funkci QESEM od Qedmy můžeš použít ke snadnému odhadování a spouštění svých obvodů s potlačením a mitigací chyb, čímž dosáhneš větších objemů obvodů a vyšší přesnosti. Pro použití QESEM zadáš kvantový Circuit, sadu pozorovatelných veličin k měření, cílovou statistickou přesnost pro každou pozorovatelnou veličinu a zvolený QPU. Před spuštěním obvodu na cílovou přesnost můžeš odhadnout požadovaný čas QPU na základě analytického výpočtu, který nevyžaduje spuštění obvodu. Jakmile jsi spokojený s odhadem času QPU, můžeš Circuit spustit s QESEM.

Při spuštění obvodu provede QESEM protokol charakterizace zařízení přizpůsobený tvému obvodu, čímž vznikne spolehlivý šumový model pro chyby vyskytující se v obvodu. Na základě charakterizace QESEM nejprve implementuje transpilaci s ohledem na šum, která mapuje vstupní Circuit na sadu fyzických Qubitů a Gate, čímž minimalizuje šum ovlivňující cílovou pozorovatelnou veličinu. Patří sem nativně dostupné Gate (CX/CZ na zařízeních IBM&reg;) i další Gate optimalizované QESEM, tvořící rozšířenou sadu Gate systému QESEM. Poté QESEM spustí na QPU sadu ES a EM obvodů založených na charakterizaci a shromáždí výsledky měření. Ty jsou následně klasicky post-procesovány a poskytují nestrannou střední hodnotu a chybové pásmo pro každou pozorovatelnou veličinu odpovídající požadované přesnosti.

![Přehled Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM prokázal schopnost poskytovat vysoce přesné výsledky pro různé kvantové aplikace a pro největší objemy obvodů dosažitelné dnes. QESEM nabízí následující funkce pro uživatele, demonstrované v části s benchmarky níže:
-	**Zaručená přesnost:** QESEM poskytuje nestranné odhady středních hodnot pozorovatelných veličin. Jeho metoda EM je vybavena teoretickými zárukami, které – společně s nejmodernější charakterizací Qedmy – zajišťují, že mitigace konverguje k výstupu bezchybného obvodu s přesností specifikovanou uživatelem. Na rozdíl od mnoha heuristických metod EM, které jsou náchylné k systematickým chybám nebo zkreslení, je zaručená přesnost QESEM nezbytná pro zajištění spolehlivých výsledků u obecných kvantových obvodů a pozorovatelných veličin.
-	**Škálovatelnost na velké QPU:** Čas QPU systému QESEM závisí na objemech obvodů, ale jinak je nezávislý na počtu Qubitů. Qedma demonstrovala QESEM na největších kvantových zařízeních dostupných dnes, včetně IBM Quantum 127-qubitového Eagle a 133-qubitového Heron zařízení.
-	**Nezávislost na aplikaci:** QESEM byl demonstrován na různých aplikacích, včetně Hamiltonovy simulace, VQE, QAOA a amplitudového odhadování. Uživatelé mohou zadat libovolný kvantový Circuit a pozorovatelnou veličinu k měření a získat přesné výsledky bez chyb. Jediná omezení jsou dána hardwarovými specifikacemi a přiděleným časem QPU, které určují přístupné objemy obvodů a přesnosti výstupu. Naproti tomu mnohá řešení pro redukci chyb jsou specifická pro konkrétní aplikaci nebo zahrnují nekontrolované heuristiky, čímž jsou nepoužitelná pro obecné kvantové obvody a aplikace.
-  **Rozšířená sada Gate:** QESEM podporuje Gate s frakčními úhly a poskytuje Qedmou optimalizované frakčně-úhlové $Rzz(\theta)$ Gate na zařízeních IBM Quantum Heron a Eagle. Tato rozšířená sada Gate umožňuje efektivnější kompilaci a odemyká objemy obvodů větší až o faktor 2 ve srovnání s výchozí kompilací CX/CZ.
-	**Multibase pozorovatelné veličiny:** QESEM podporuje vstupní pozorovatelné veličiny složené z mnoha nekomutujících Pauliho řetězců, jako jsou obecné Hamiltonovy operátory. Výběr měřicích bází a optimalizace alokace QPU zdrojů (snímky a obvody) je pak prováděna automaticky systémem QESEM tak, aby minimalizovala požadovaný čas QPU pro požadovanou přesnost. Tato optimalizace, která bere v úvahu věrnosti hardwaru a rychlosti spouštění, ti umožňuje spouštět hlubší obvody a dosahovat vyšší přesnosti.
## Benchmarky
QESEM byl testován na široké škále případů použití a aplikací. Následující příklady ti pomohou posoudit, jaké typy pracovních zátěží lze s QESEM spouštět.

Klíčovým ukazatelem pro kvantifikaci náročnosti jak mitigace chyb, tak klasické simulace pro daný Circuit a pozorovatelnou veličinu je **aktivní objem**: počet CNOT Gate ovlivňujících pozorovatelnou veličinu v obvodu. Aktivní objem závisí na hloubce a šířce obvodu, na váze pozorovatelné veličiny a na struktuře obvodu, která určuje světelný kužel pozorovatelné veličiny. Podrobnosti najdeš v přednášce z [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM poskytuje zvláště velkou hodnotu v režimu vysokého objemu, kde dává spolehlivé výsledky pro obecné obvody a pozorovatelné veličiny.

![Aktivní objem](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplikace    | Počet Qubitů | Zařízení | Popis obvodu | Přesnost | Celkový čas | Využití Runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| VQE Circuit  | 8              | Eagle (r3) | 21 celkových vrstev, 9 měřicích bází, 1D řetězec                    | 98 %      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 jedinečné vrstvy × 3 kroky, 2D topologie heavy-hex                      | 97 %     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 jedinečné vrstvy × 8 kroků, 2D topologie heavy-hex                     | 97 %      | 116 min      | 23 min          |
| Trotterizovaná Hamiltonova simulace   | 40  | Eagle (r3)            | 2 jedinečné vrstvy × 10 Trotterových kroků, 1D řetězec                    | 97 %      | 3 hodiny     | 25 min         |
| Trotterizovaná Hamiltonova simulace   | 119  | Eagle (r3)           | 3 jedinečné vrstvy × 9 Trotterových kroků, 2D topologie heavy-hex                    | 95 %      | 6,5 hodiny     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 jedinečné vrstvy × 15 kroků, 2D topologie heavy-hex                    | 99 %      | 52 min             | 9 min           |

Přesnost je zde měřena relativně k ideální hodnotě pozorovatelné veličiny: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, kde '$\epsilon$' je absolutní přesnost mitigace (nastavená vstupem uživatele) a $\langle O \rangle_{ideal}$ je pozorovatelná veličina bezchybného obvodu.
„Využití Runtime" měří využití benchmarku v dávkovém režimu (součet využití jednotlivých úloh), zatímco „celkový čas" měří využití v session režimu (celková doba experimentu), která zahrnuje dodatečné klasické a komunikační časy. QESEM je dostupný pro spouštění v obou režimech, takže uživatelé mohou co nejlépe využít své dostupné zdroje.

28-qubitové obvody Kicked Ising simulují Diskrétní časový kvazikrystal studovaný Shinjem et al. (viz [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) a [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) na třech propojených smyčkách ibm_kawasaki. Parametry obvodu použité zde jsou $(\theta_x, \theta_z) = (0.9 \pi, 0)$ s feromagnetickým počátečním stavem $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Měřená pozorovatelná veličina je absolutní hodnota magnetizace $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Experiment Kicked Ising v měřítku užitečnosti byl spuštěn na 136 nejlepších Qubitech ibm_fez; tento konkrétní benchmark byl spuštěn na Cliffordově úhlu $(\theta_x, \theta_z) = (\pi, 0)$, při němž aktivní objem roste s hloubkou obvodu pomalu, což – společně s vysokou věrností zařízení – umožňuje vysokou přesnost při krátkém čase běhu.

Trotterizované obvody Hamiltonovy simulace jsou pro model Ising s příčným polem při frakčních úhlech: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ a $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ (viz [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Obvod v měřítku užitečnosti byl spuštěn na 119 nejlepších Qubitech ibm_brisbane, zatímco experiment se 40 Qubity byl spuštěn na nejlepším dostupném řetězci. Přesnost je uváděna pro magnetizaci; vysoce přesné výsledky byly získány i pro pozorovatelné veličiny s vyšší vahou.

Obvod VQE byl vyvinut společně s výzkumníky z Centra pro kvantové technologie a aplikace Německého elektronu-synchrotronu (DESY). Cílová pozorovatelná veličina zde byl Hamiltonův operátor složený z velkého počtu nekomutujících Pauliho řetězců, což zdůrazňuje optimalizovaný výkon QESEM pro pozorovatelné veličiny ve více bázích. Mitigace byla aplikována na klasicky optimalizovaný ansatz; ačkoli tyto výsledky jsou zatím nepublikované, výsledky stejné kvality budou získány pro různé obvody s podobnými strukturálními vlastnostmi.
## Začínáme
Autentizuj se pomocí svého [IBM Quantum Platform API klíče](http://quantum.cloud.ibm.com/) a vyber Qiskit Function QESEM takto. (Tento úryvek předpokládá, že jsi již [uložil svůj účet](/guides/functions#install-qiskit-functions-catalog-client) do svého lokálního prostředí.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Pro kontrolu stavu pracovní zátěže Qiskit Function nebo získání výsledků můžeš použít známá API Qiskit Serverless:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Následující úryvek kódu popisuje, jak získat odhad doby QPU (když je nastaveno `estimate_time_only`):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


Následující úryvek kódu ukazuje, jak získat výsledky mitigace (pokud není nastaveno `estimate_time_only`) a metriky provádění. Tyto obsahují základní data, která umožňují hlubší pochopení toho, jak různé parametry ovlivňují provádění QESEM. Mohou být také relevantní při psaní článku na základě tvého výzkumu.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Načtení chybových zpráv
Pokud má tvoje úloha stav ERROR, použij `job.result()` k načtení chybové zprávy takto:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Získání podpory

Tým podpory Qedmy je tu, aby ti pomohl! Pokud narazíš na jakékoli problémy nebo máš otázky ohledně používání Qiskit Function QESEM, neváhej nás kontaktovat. Naši znalí a přátelští pracovníci podpory jsou připraveni ti pomoci s jakýmikoli technickými obavami nebo dotazy.

Můžeš nám napsat e-mail na support@qedma.com a požádat o pomoc. Uveď prosím co nejvíce podrobností o problému, který máš, abychom ti mohli poskytnout rychlou a přesnou odpověď. Můžeš také kontaktovat svého dedikovaného zástupce Qedmy (POC) e-mailem nebo telefonem.

Abychom ti mohli pomoci efektivněji, uveď prosím při kontaktu s námi následující informace:

- Podrobný popis problému
- ID úlohy
- Všechny relevantní chybové zprávy nebo kódy

Jsme odhodláni poskytnout ti rychlou a účinnou podporu, aby tvé zkušenosti s naší Qiskit Function byly co nejlepší.

Vždy hledáme způsoby, jak náš produkt vylepšit, a vítáme tvé návrhy! Pokud máš nápady, jak bychom mohli vylepšit naše služby nebo jaké funkce bys chtěl/a vidět, pošli nám své myšlenky na support@qedma.com.

## Další kroky

> **Tip:** - [Požádat o přístup k Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Navštiv [referenci API](https://docs.quantum.ibm.com/api/functions/qedma-qesem) pro tuto Qiskit Function.
> - Prostuduj [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Prostuduj [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).